In [1]:
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display


Matplotlib is building the font cache; this may take a moment.


In [2]:
balance_widget = widgets.ToggleButtons(
    options=["True", "False"],
    description='Balance:',
    value="True"
)

pca_widget = widgets.ToggleButtons(
    options=["4", "8"],
    description='PCA:',
    value="4"
)

percent_widget = widgets.ToggleButtons(
    options=["0.1", "0.25", "0.5", "0.75", "1"],
    description='Dataset %:',
    value="1"
)

# controls = widgets.VBox([balance_widget, pca_widget, percent_widget])
# display(controls)

In [3]:
here = Path.cwd()


In [4]:
results = Path(here / "Thompson_QSVM_results_20260224")

In [5]:
for file in [f for f in results.iterdir() if f.is_file()]:
    print(file.name)


False_8_0.25_CCM_train.txt
True_4_1_CCM_train.txt
True_8_0.5_MCC.txt
4pca_vs_8pca_violin_1_balFalse.png
False_4_0.75_MCC_scores.txt
True_4_1_costs.txt
True_4_0.75_CCM_train.txt
False_4_0.1_val_TC_savings.txt
False_4_0.5_train_pred_TC_mcc.txt
True_4_0.25_MCC.txt
quantum_MCC_by_size_True_8.png
True_8_0.75_savings_summary.txt
False_4_0.1_MCC_scores.txt
True_4_1_val_TC_mcc.txt
False_8_0.5_classical_TC_mcc.txt
4pca_vs_8pca_1_balTrue.png
False_8_0.1_train_pred_TC_mcc.txt
True_8_0.75_train_pred_TC_mcc.txt
bal_vs_unbal_FinallMCC_8_1.png
False_4_0.25_train_pred_TC_mcc.txt
4pca_vs_8pca_violin_1_balTrue.png
False_4_0.5_MCC.txt
False_4_0.25_CCM_val.txt
True_8_0.1_MCC_scores.txt
True_4_0.1_MCC_scores.txt
False_4_1_MCC.txt
True_4_0.25_CCM_train.txt
Final_Confusion_1_8_balFalse.png
True_8_0.1_classical_TC_mcc.txt
True_8_1_val_TC_savings.txt
True_8_0.1_savings_summary.txt
True_8_0.25_val_TC_savings.txt
True_4_0.25_classical_TC_savings.txt
True_4_0.1_MCC.txt
True_4_0.1_train_pred_TC_savings.txt
False_8

In [6]:
True_4_1_costs = np.loadtxt(results / "True_4_1_costs.txt")

In [7]:
data_dict = {
    file.name.removesuffix(".txt"): np.loadtxt(file)
    for file in results.iterdir()
    if file.is_file() and file.name.endswith(".txt")
}

In [8]:
True_4_1_costs.shape

(10, 50)

# balanced vs unbalanced plot

In [9]:
def balanced_vs_unbalanced_violin(pca, percent):
    print(f"PCA: {pca}, Percent: {percent}")
    plt.clf()
    
    # Load the data
    Q_val_balanced = pd.DataFrame(data_dict[f"True_{pca}_{percent}_QCM_val"],
                                  columns=["True Healthy", "False Falure", "False Healthy", "True Failure"])
    Q_val_balanced['Set'] = 'Quantum Validation Balanced'
    
    Q_val_unbalanced = pd.DataFrame(data_dict[f"False_{pca}_{percent}_QCM_val"],
                                    columns=["True Healthy", "False Falure", "False Healthy", "True Failure"])
    Q_val_unbalanced['Set'] = 'Quantum Validation Unbalanced'
    
    C_val_balanced = pd.DataFrame(data_dict[f"True_{pca}_{percent}_CCM_val"],
                                  columns=["True Healthy", "False Falure", "False Healthy", "True Failure"])
    C_val_balanced['Set'] = 'Classical Validation Balanced'
    
    C_val_unbalanced = pd.DataFrame(data_dict[f"False_{pca}_{percent}_CCM_val"],
                                    columns=["True Healthy", "False Falure", "False Healthy", "True Failure"])
    C_val_unbalanced['Set'] = 'Classical Validation Unbalanced'

    # Combine the data
    df = pd.concat([Q_val_balanced, Q_val_unbalanced, C_val_balanced, C_val_unbalanced], ignore_index=True)
    df_melted = df.melt(id_vars='Set', var_name='Metric', value_name='Value')
    
    plt.figure(figsize=(12, 6))
    sns.violinplot(x='Metric', y='Value', hue='Set', data=df_melted, split=True)
    plt.xlabel("Confusion Matrix Entries")
    plt.ylabel("Relative Frequency")
    plt.title(f'Violin Plot: Balanced vs Unbalanced for {float(percent)*100}% of the data with {pca} components')
    plt.savefig(results / f"balanced_vs_unbalanced_violin_{pca}_{percent}.png")
    plt.show()
    


    # Same thing but with the training curve
    classical_bal = data_dict[f"True_{pca}_{percent}_classical_TC_savings"]
    training_bal = data_dict[f"True_{pca}_{percent}_train_pred_TC_savings"]
    val_bal = data_dict[f"True_{pca}_{percent}_val_TC_savings"]
    
    plt.plot(np.arange(0, training_bal.shape[1]), np.mean(training_bal, 0), label="training balanced")
    plt.fill_between(np.arange(0, training_bal.shape[1]), 
                     np.mean(training_bal, 0) - (1.96*np.std(training_bal, 0)/np.sqrt(training_bal.shape[0])), 
                     np.mean(training_bal, 0) + (1.96*np.std(training_bal, 0)/np.sqrt(training_bal.shape[0])),
                     alpha=0.2)
    
    plt.plot(np.arange(0, val_bal.shape[1]), np.mean(val_bal, 0), label="validation balanced")
    plt.fill_between(np.arange(0, val_bal.shape[1]), 
                     np.mean(val_bal, 0) - (1.96*np.std(val_bal, 0)/np.sqrt(val_bal.shape[0])), 
                     np.mean(val_bal, 0) + (1.96*np.std(val_bal, 0)/np.sqrt(val_bal.shape[0])),
                     alpha=0.2)

    plt.plot(np.arange(0, classical_bal.shape[1]), np.mean(classical_bal, 0), label="classical balanced")
    plt.fill_between(np.arange(0, classical_bal.shape[1]), 
                     np.mean(classical_bal, 0) - (1.96*np.std(classical_bal, 0)/np.sqrt(classical_bal.shape[0])), 
                     np.mean(classical_bal, 0) + (1.96*np.std(classical_bal, 0)/np.sqrt(classical_bal.shape[0])),
                     alpha=0.2)
    classical_unbal = data_dict[f"False_{pca}_{percent}_classical_TC_savings"]
    training_unbal = data_dict[f"False_{pca}_{percent}_train_pred_TC_savings"]
    val_unbal = data_dict[f"False_{pca}_{percent}_val_TC_savings"]
    
    plt.plot(np.arange(0, training_unbal.shape[1]), np.mean(training_unbal, 0), label="training unbalanced", linestyle="--")
    plt.fill_between(np.arange(0, training_unbal.shape[1]), 
                     np.mean(training_unbal, 0) - (1.96*np.std(training_unbal, 0)/np.sqrt(training_unbal.shape[0])), 
                     np.mean(training_unbal, 0) + (1.96*np.std(training_unbal, 0)/np.sqrt(training_unbal.shape[0])),
                     alpha=0.2)
    
    plt.plot(np.arange(0, val_unbal.shape[1]), np.mean(val_unbal, 0), label="validation unbalanced", linestyle="--")
    plt.fill_between(np.arange(0, val_unbal.shape[1]), 
                     np.mean(val_unbal, 0) - (1.96*np.std(val_unbal, 0)/np.sqrt(val_unbal.shape[0])), 
                     np.mean(val_unbal, 0) + (1.96*np.std(val_unbal, 0)/np.sqrt(val_unbal.shape[0])),
                     alpha=0.2)

    plt.plot(np.arange(0, classical_unbal.shape[1]), np.mean(classical_unbal, 0), label="classical unbalanced", linestyle="--")
    plt.fill_between(np.arange(0, classical_unbal.shape[1]), 
                     np.mean(classical_unbal, 0) - (1.96*np.std(classical_unbal, 0)/np.sqrt(classical_unbal.shape[0])), 
                     np.mean(classical_unbal, 0) + (1.96*np.std(classical_unbal, 0)/np.sqrt(classical_unbal.shape[0])),
                     alpha=0.2)

    
    plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
    plt.title(f"Balanced vs unbalanced training curve for {float(percent)*100}% of the data with {pca} components")
    plt.xlabel("Number of iterations")
    plt.ylabel("savings")
    plt.savefig(results / f"balanced_vs_unbalanced_{pca}_{percent}.png", bbox_inches='tight', pad_inches=0.1)
    plt.show()

    
    #Save thing but MCC
    classical_bal = data_dict[f"True_{pca}_{percent}_classical_TC_mcc"]
    training_bal = data_dict[f"True_{pca}_{percent}_train_pred_TC_mcc"]
    val_bal = data_dict[f"True_{pca}_{percent}_val_TC_mcc"]
    
    plt.plot(np.arange(0, training_bal.shape[1]), np.mean(training_bal, 0), label="training balanced")
    plt.fill_between(np.arange(0, training_bal.shape[1]), 
                     np.mean(training_bal, 0) - (1.96*np.std(training_bal, 0)/np.sqrt(training_bal.shape[0])), 
                     np.mean(training_bal, 0) + (1.96*np.std(training_bal, 0)/np.sqrt(training_bal.shape[0])),
                     alpha=0.2)
    
    plt.plot(np.arange(0, val_bal.shape[1]), np.mean(val_bal, 0), label="validation balanced")
    plt.fill_between(np.arange(0, val_bal.shape[1]), 
                     np.mean(val_bal, 0) - (1.96*np.std(val_bal, 0)/np.sqrt(val_bal.shape[0])), 
                     np.mean(val_bal, 0) + (1.96*np.std(val_bal, 0)/np.sqrt(val_bal.shape[0])),
                     alpha=0.2)

    plt.plot(np.arange(0, classical_bal.shape[1]), np.mean(classical_bal, 0), label="classical balanced")
    plt.fill_between(np.arange(0, classical_bal.shape[1]), 
                     np.mean(classical_bal, 0) - (1.96*np.std(classical_bal, 0)/np.sqrt(classical_bal.shape[0])), 
                     np.mean(classical_bal, 0) + (1.96*np.std(classical_bal, 0)/np.sqrt(classical_bal.shape[0])),
                     alpha=0.2)
    classical_unbal = data_dict[f"False_{pca}_{percent}_classical_TC_mcc"]
    training_unbal = data_dict[f"False_{pca}_{percent}_train_pred_TC_mcc"]
    val_unbal = data_dict[f"False_{pca}_{percent}_val_TC_mcc"]
    
    plt.plot(np.arange(0, training_unbal.shape[1]), np.mean(training_unbal, 0), label="training unbalanced", linestyle="--")
    plt.fill_between(np.arange(0, training_unbal.shape[1]), 
                     np.mean(training_unbal, 0) - (1.96*np.std(training_unbal, 0)/np.sqrt(training_unbal.shape[0])), 
                     np.mean(training_unbal, 0) + (1.96*np.std(training_unbal, 0)/np.sqrt(training_unbal.shape[0])),
                     alpha=0.2)
    
    plt.plot(np.arange(0, val_unbal.shape[1]), np.mean(val_unbal, 0), label="validation unbalanced", linestyle="--")
    plt.fill_between(np.arange(0, val_unbal.shape[1]), 
                     np.mean(val_unbal, 0) - (1.96*np.std(val_unbal, 0)/np.sqrt(val_unbal.shape[0])), 
                     np.mean(val_unbal, 0) + (1.96*np.std(val_unbal, 0)/np.sqrt(val_unbal.shape[0])),
                     alpha=0.2)

    plt.plot(np.arange(0, classical_unbal.shape[1]), np.mean(classical_unbal, 0), label="classical unbalanced", linestyle="--")
    plt.fill_between(np.arange(0, classical_unbal.shape[1]), 
                     np.mean(classical_unbal, 0) - (1.96*np.std(classical_unbal, 0)/np.sqrt(classical_unbal.shape[0])), 
                     np.mean(classical_unbal, 0) + (1.96*np.std(classical_unbal, 0)/np.sqrt(classical_unbal.shape[0])),
                     alpha=0.2)

    
    plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
    plt.title(f"Balanced vs unbalanced MCC accross training epochs for {float(percent)*100}% of the data with {pca} components")
    plt.xlabel("Number of iterations")
    plt.ylabel("MCC Score")
    plt.savefig(results / f"balanced_vs_unbalanced_mcc_{pca}_{percent}.png", bbox_inches='tight', pad_inches=0.1)
    plt.show()


    # Final MCC Scores
    df = pd.DataFrame(data_dict[f"True_{pca}_{percent}_MCC_scores"])
    
    Q_MCC_bal = df.iloc[:, :2].copy()
    Q_MCC_bal.columns = ['training', 'validation']
    Q_MCC_bal['Set'] = 'Quantum bal'

    C_MCC_bal = df.iloc[:, 2:4].copy()
    C_MCC_bal.columns = ['training', 'validation']
    C_MCC_bal['Set'] = 'Classical bal'

    df = pd.DataFrame(data_dict[f"False_{pca}_{percent}_MCC_scores"])
    
    Q_MCC_unbal = df.iloc[:, :2].copy()
    Q_MCC_unbal.columns = ['training', 'validation']
    Q_MCC_unbal['Set'] = 'Quantum unbal'

    C_MCC_unbal = df.iloc[:, 2:4].copy()
    C_MCC_unbal.columns = ['training', 'validation']
    C_MCC_unbal['Set'] = 'Classical unbal'


    # Combine the data
    df = pd.concat([Q_MCC_bal, Q_MCC_unbal, C_MCC_bal, C_MCC_unbal], ignore_index=True)
    df_melted = df.melt(id_vars='Set', var_name='Metric', value_name='Value')
    
    plt.figure(figsize=(12, 6))
    sns.violinplot(x='Metric', y='Value', hue='Set', data=df_melted, split=True)
    plt.xlabel("")
    plt.ylabel("MCC Score")
    plt.title(f'Violin Plot: bal vs unbal for {float(percent)*100}% of the data with {pca} PCA components')
    plt.savefig(results / f"bal_vs_unbal_FinallMCC_{pca}_{percent}.png")
    plt.show()


display(widgets.interactive(
    balanced_vs_unbalanced_violin,
    pca=pca_widget,
    percent=percent_widget
))

interactive(children=(ToggleButtons(description='PCA:', options=('4', '8'), value='4'), ToggleButtons(descript…

# 4 vs 8 PCA

In [10]:
def pca_comparison(balance = True, percent = 1):
    print(f"balance: {balance}, Percent: {percent}")
    plt.clf()
    
    # Load the data
    Q_val_4pca = pd.DataFrame(data_dict[f"{balance}_4_{percent}_QCM_val"],
                                  columns=["True Healthy", "False Falure", "False Healthy", "True Failure"])
    Q_val_4pca['Set'] = 'Quantum Validation 4pca'
    
    Q_val_8pca = pd.DataFrame(data_dict[f"{balance}_8_{percent}_QCM_val"],
                                    columns=["True Healthy", "False Falure", "False Healthy", "True Failure"])
    Q_val_8pca['Set'] = 'Quantum Validation 8pca'
    
    C_val_4pca = pd.DataFrame(data_dict[f"{balance}_4_{percent}_CCM_val"],
                                  columns=["True Healthy", "False Falure", "False Healthy", "True Failure"])
    C_val_4pca['Set'] = 'Classical Validation 4pca'
    
    C_val_8pca = pd.DataFrame(data_dict[f"{balance}_8_{percent}_CCM_val"],
                                    columns=["True Healthy", "False Falure", "False Healthy", "True Failure"])
    C_val_8pca['Set'] = 'Classical Validation 8pca'

    # Combine the data
    df = pd.concat([Q_val_4pca, Q_val_8pca, C_val_4pca, C_val_8pca], ignore_index=True)
    df_melted = df.melt(id_vars='Set', var_name='Metric', value_name='Value')
    
    plt.figure(figsize=(12, 6))
    sns.violinplot(x='Metric', y='Value', hue='Set', data=df_melted, split=True)
    plt.xlabel("Confusion Matrix Entries")
    plt.ylabel("Relative Frequency")
    plt.title(f'Violin Plot: 4pca vs 8pca for {float(percent)*100}% of the data with balanced = {balance}')
    plt.savefig(results / f"4pca_vs_8pca_violin_{percent}_bal{balance}.png")
    plt.show()
    


 # Same thing but with the training curve
    classical_4pca = data_dict[f"{balance}_4_{percent}_classical_TC_savings"]
    training_4pca = data_dict[f"{balance}_4_{percent}_train_pred_TC_savings"]
    val_4pca = data_dict[f"{balance}_4_{percent}_val_TC_savings"]
    
    plt.plot(np.arange(0, training_4pca.shape[1]), np.mean(training_4pca, 0), label="training 4pca")
    plt.fill_between(np.arange(0, training_4pca.shape[1]), 
                     np.mean(training_4pca, 0) - (1.96*np.std(training_4pca, 0)/np.sqrt(training_4pca.shape[0])), 
                     np.mean(training_4pca, 0) + (1.96*np.std(training_4pca, 0)/np.sqrt(training_4pca.shape[0])),
                     alpha=0.2)
    
    plt.plot(np.arange(0, val_4pca.shape[1]), np.mean(val_4pca, 0), label="validation 4pca")
    plt.fill_between(np.arange(0, val_4pca.shape[1]), 
                     np.mean(val_4pca, 0) - (1.96*np.std(val_4pca, 0)/np.sqrt(val_4pca.shape[0])), 
                     np.mean(val_4pca, 0) + (1.96*np.std(val_4pca, 0)/np.sqrt(val_4pca.shape[0])),
                     alpha=0.2)

    plt.plot(np.arange(0, classical_4pca.shape[1]), np.mean(classical_4pca, 0), label="classical 4pca")
    plt.fill_between(np.arange(0, classical_4pca.shape[1]), 
                     np.mean(classical_4pca, 0) - (1.96*np.std(classical_4pca, 0)/np.sqrt(classical_4pca.shape[0])), 
                     np.mean(classical_4pca, 0) + (1.96*np.std(classical_4pca, 0)/np.sqrt(classical_4pca.shape[0])),
                     alpha=0.2)
    
    classical_8pca = data_dict[f"{balance}_8_{percent}_classical_TC_savings"]
    training_8pca = data_dict[f"{balance}_8_{percent}_train_pred_TC_savings"]
    val_8pca = data_dict[f"{balance}_8_{percent}_val_TC_savings"]
    
    plt.plot(np.arange(0, training_8pca.shape[1]), np.mean(training_8pca, 0), label="training 8pca", linestyle="--")
    plt.fill_between(np.arange(0, training_8pca.shape[1]), 
                     np.mean(training_8pca, 0) - (1.96*np.std(training_8pca, 0)/np.sqrt(training_8pca.shape[0])), 
                     np.mean(training_8pca, 0) + (1.96*np.std(training_8pca, 0)/np.sqrt(training_8pca.shape[0])),
                     alpha=0.2)
    
    plt.plot(np.arange(0, val_8pca.shape[1]), np.mean(val_8pca, 0), label="validation 8pca", linestyle="--")
    plt.fill_between(np.arange(0, val_8pca.shape[1]), 
                     np.mean(val_8pca, 0) - (1.96*np.std(val_8pca, 0)/np.sqrt(val_8pca.shape[0])), 
                     np.mean(val_8pca, 0) + (1.96*np.std(val_8pca, 0)/np.sqrt(val_8pca.shape[0])),
                     alpha=0.2)

    plt.plot(np.arange(0, classical_8pca.shape[1]), np.mean(classical_8pca, 0), label="classical 8pca", linestyle="--")
    plt.fill_between(np.arange(0, classical_8pca.shape[1]), 
                     np.mean(classical_8pca, 0) - (1.96*np.std(classical_8pca, 0)/np.sqrt(classical_8pca.shape[0])), 
                     np.mean(classical_8pca, 0) + (1.96*np.std(classical_8pca, 0)/np.sqrt(classical_8pca.shape[0])),
                     alpha=0.2)

    
    plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
    plt.title(f"4pca vs 8pca training curve for {float(percent)*100}% of the data with balance = {balance}")
    plt.xlabel("Number of iterations")
    plt.ylabel("savings")
    plt.savefig(results / f"4pca_vs_8pca_{percent}_bal{balance}.png", bbox_inches='tight', pad_inches=0.1)
    plt.show()

    #MCC Curves
    classical_4pca = data_dict[f"{balance}_4_{percent}_classical_TC_mcc"]
    training_4pca = data_dict[f"{balance}_4_{percent}_train_pred_TC_mcc"]
    val_4pca = data_dict[f"{balance}_4_{percent}_val_TC_mcc"]
    
    plt.plot(np.arange(0, training_4pca.shape[1]), np.mean(training_4pca, 0), label="training 4pca")
    plt.fill_between(np.arange(0, training_4pca.shape[1]), 
                     np.mean(training_4pca, 0) - (1.96*np.std(training_4pca, 0)/np.sqrt(training_4pca.shape[0])), 
                     np.mean(training_4pca, 0) + (1.96*np.std(training_4pca, 0)/np.sqrt(training_4pca.shape[0])),
                     alpha=0.2)
    
    plt.plot(np.arange(0, val_4pca.shape[1]), np.mean(val_4pca, 0), label="validation 4pca")
    plt.fill_between(np.arange(0, val_4pca.shape[1]), 
                     np.mean(val_4pca, 0) - (1.96*np.std(val_4pca, 0)/np.sqrt(val_4pca.shape[0])), 
                     np.mean(val_4pca, 0) + (1.96*np.std(val_4pca, 0)/np.sqrt(val_4pca.shape[0])),
                     alpha=0.2)

    plt.plot(np.arange(0, classical_4pca.shape[1]), np.mean(classical_4pca, 0), label="classical 4pca")
    plt.fill_between(np.arange(0, classical_4pca.shape[1]), 
                     np.mean(classical_4pca, 0) - (1.96*np.std(classical_4pca, 0)/np.sqrt(classical_4pca.shape[0])), 
                     np.mean(classical_4pca, 0) + (1.96*np.std(classical_4pca, 0)/np.sqrt(classical_4pca.shape[0])),
                     alpha=0.2)
    
    classical_8pca = data_dict[f"{balance}_8_{percent}_classical_TC_mcc"]
    training_8pca = data_dict[f"{balance}_8_{percent}_train_pred_TC_mcc"]
    val_8pca = data_dict[f"{balance}_8_{percent}_val_TC_mcc"]
    
    plt.plot(np.arange(0, training_8pca.shape[1]), np.mean(training_8pca, 0), label="training 8pca", linestyle="--")
    plt.fill_between(np.arange(0, training_8pca.shape[1]), 
                     np.mean(training_8pca, 0) - (1.96*np.std(training_8pca, 0)/np.sqrt(training_8pca.shape[0])), 
                     np.mean(training_8pca, 0) + (1.96*np.std(training_8pca, 0)/np.sqrt(training_8pca.shape[0])),
                     alpha=0.2)
    
    plt.plot(np.arange(0, val_8pca.shape[1]), np.mean(val_8pca, 0), label="validation 8pca", linestyle="--")
    plt.fill_between(np.arange(0, val_8pca.shape[1]), 
                     np.mean(val_8pca, 0) - (1.96*np.std(val_8pca, 0)/np.sqrt(val_8pca.shape[0])), 
                     np.mean(val_8pca, 0) + (1.96*np.std(val_8pca, 0)/np.sqrt(val_8pca.shape[0])),
                     alpha=0.2)

    plt.plot(np.arange(0, classical_8pca.shape[1]), np.mean(classical_8pca, 0), label="classical 8pca", linestyle="--")
    plt.fill_between(np.arange(0, classical_8pca.shape[1]), 
                     np.mean(classical_8pca, 0) - (1.96*np.std(classical_8pca, 0)/np.sqrt(classical_8pca.shape[0])), 
                     np.mean(classical_8pca, 0) + (1.96*np.std(classical_8pca, 0)/np.sqrt(classical_8pca.shape[0])),
                     alpha=0.2)

    
    plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
    plt.title(f"4pca vs 8pca MCC across training epoches for {float(percent)*100}% of the data with balance = {balance}")
    plt.xlabel("Number of iterations")
    plt.ylabel("MCC Score")
    plt.savefig(results / f"4pca_vs_8pca_mccTC_{percent}_bal{balance}.png", bbox_inches='tight', pad_inches=0.1)
    plt.show()

    # Final MCC Scores
    df = pd.DataFrame(data_dict[f"{balance}_4_{percent}_MCC_scores"])
    
    Q_MCC_4pca = df.iloc[:, :2].copy()
    Q_MCC_4pca.columns = ['training', 'validation']
    Q_MCC_4pca['Set'] = 'Quantum 4pca'

    C_MCC_4pca = df.iloc[:, 2:4].copy()
    C_MCC_4pca.columns = ['training', 'validation']
    C_MCC_4pca['Set'] = 'Classical 4pca'

    df = pd.DataFrame(data_dict[f"{balance}_8_{percent}_MCC_scores"])
    
    Q_MCC_8pca = df.iloc[:, :2].copy()
    Q_MCC_8pca.columns = ['training', 'validation']
    Q_MCC_8pca['Set'] = 'Quantum 8pca'

    C_MCC_8pca = df.iloc[:, 2:4].copy()
    C_MCC_8pca.columns = ['training', 'validation']
    C_MCC_8pca['Set'] = 'Classical 8pca'


    # Combine the data
    df = pd.concat([Q_MCC_4pca, Q_MCC_8pca, C_MCC_4pca, C_MCC_8pca], ignore_index=True)
    df_melted = df.melt(id_vars='Set', var_name='Metric', value_name='Value')
    
    plt.figure(figsize=(12, 6))
    sns.violinplot(x='Metric', y='Value', hue='Set', data=df_melted, split=True)
    plt.xlabel("")
    plt.ylabel("MCC Score")
    plt.title(f'Violin Plot: 4pca vs 8pca for {float(percent)*100}% of the data with balanced = {balance}')
    plt.savefig(results / f"4pca_vs_8pca_FinallMCC_{percent}_bal{balance}.png")
    plt.show()

    
    

display(widgets.interactive(
    pca_comparison,
    balance=balance_widget,
    percent=percent_widget
))

interactive(children=(ToggleButtons(description='Balance:', options=('True', 'False'), value='True'), ToggleBu…

# dataset size comparison

In [11]:
def size_comp(balance=True, pca=4):
    print(f"balance: {balance}, PCA Components: {pca}")
    plt.clf()
    
    sizes = [0.1, 0.25, 0.5, 0.75, 1]
    size_labels = ['10%', '25%', '50%', '75%', '100%']
    classical_rows = []
    quantum_rows = []
    
    for size, label in zip(sizes, size_labels):
        df = pd.DataFrame(data_dict[f"{balance}_{pca}_{size}_MCC_scores"])
        # Training
        quantum_rows.append({'Set': 'Training', 'Size': label, 'Value': df.iloc[:,0]})
        classical_rows.append({'Set': 'Training', 'Size': label, 'Value': df.iloc[:,2]})
        # Validation
        quantum_rows.append({'Set': 'Validation', 'Size': label, 'Value': df.iloc[:,1]})
        classical_rows.append({'Set': 'Validation', 'Size': label, 'Value': df.iloc[:,3]})
    
    # Flatten lists
    quantum_df = pd.concat([
        pd.DataFrame({'Set': row['Set'], 'Size': row['Size'], 'Value': row['Value']})
        for row in quantum_rows
    ], ignore_index=True)
    classical_df = pd.concat([
        pd.DataFrame({'Set': row['Set'], 'Size': row['Size'], 'Value': row['Value']})
        for row in classical_rows
    ], ignore_index=True)
    
    # Classical plot
    plt.figure(figsize=(8, 6))
    sns.violinplot(x='Size', y='Value', hue='Set', data=classical_df, split=True)
    plt.xlabel("Dataset Size")
    plt.ylabel("MCC Score")
    plt.title('Classical MCC by Dataset Size')
    plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.savefig(results / f"classical_MCC_by_size_{balance}_{pca}.png")
    plt.show()
    
    # Quantum plot
    plt.figure(figsize=(8, 6))
    sns.violinplot(x='Size', y='Value', hue='Set', data=quantum_df, split=True)
    plt.xlabel("Dataset Size")
    plt.ylabel("MCC Score")
    plt.title('Quantum MCC by Dataset Size')
    plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.savefig(results / f"quantum_MCC_by_size_{balance}_{pca}.png")
    plt.show()

display(widgets.interactive(
    size_comp,
    balance=balance_widget,
    pca=pca_widget
))

interactive(children=(ToggleButtons(description='Balance:', options=('True', 'False'), value='True'), ToggleBu…

# Final Plots

In [12]:
def itteration_plots(balance, pca, percent):
    set_name = f"{balance}_{pca}_{percent}"
    print(f"Set name: {set_name}")

    classical = data_dict[f"{set_name}_classical_TC_savings"]
    training = data_dict[f"{set_name}_train_pred_TC_savings"]
    val = data_dict[f"{set_name}_val_TC_savings"]
    
    plt.plot(np.arange(0, training.shape[1]), np.mean(training, 0), label="training")
    plt.fill_between(np.arange(0, training.shape[1]), 
                     np.mean(training, 0) - (1.96*np.std(training, 0)/np.sqrt(training.shape[0])), 
                     np.mean(training, 0) + (1.96*np.std(training, 0)/np.sqrt(training.shape[0])),
                     alpha=0.2)
    
    plt.plot(np.arange(0, val.shape[1]), np.mean(val, 0), label="val")
    plt.fill_between(np.arange(0, val.shape[1]), 
                     np.mean(val, 0) - (1.96*np.std(val, 0)/np.sqrt(val.shape[0])), 
                     np.mean(val, 0) + (1.96*np.std(val, 0)/np.sqrt(val.shape[0])),
                     alpha=0.2)

    plt.plot(np.arange(0, classical.shape[1]), np.mean(classical, 0), label="classical")
    plt.fill_between(np.arange(0, classical.shape[1]), 
                     np.mean(classical, 0) - (1.96*np.std(classical, 0)/np.sqrt(classical.shape[0])), 
                     np.mean(classical, 0) + (1.96*np.std(classical, 0)/np.sqrt(classical.shape[0])),
                     alpha=0.2)
    
    plt.legend()
    plt.title(f"Training Curve for {float(percent)*100}% of the data with {pca} components and balanced = {balance}")
    plt.xlabel("Number of iterations")
    plt.ylabel("savings")
    plt.savefig(results / f"Final_TrainingCurve_{percent}_{pca}_bal{balance}.png", bbox_inches='tight', pad_inches=0.1)
    plt.show()

    
display(widgets.interactive(
    itteration_plots,
    balance=balance_widget,
    pca=pca_widget,
    percent=percent_widget
))

interactive(children=(ToggleButtons(description='Balance:', options=('True', 'False'), value='True'), ToggleBu…

In [13]:
def confusion_violin(balance, pca, percent):
    set_name = f"{balance}_{pca}_{percent}"
    print(f"Set name: {set_name}")

    Q_val = pd.DataFrame(data_dict[set_name + "_QCM_val"],
                        columns = ["True Healthy", "False Falure", "False Healthy", "True Failure"])
    Q_val['Set'] = 'Quantum Validation'
    
    Q_train = pd.DataFrame(data_dict[set_name + "_QCM_train"],
                        columns = ["True Healthy", "False Falure", "False Healthy", "True Failure"])
    Q_train['Set'] = 'Quantum Training'
    
    C_val = pd.DataFrame(data_dict[set_name + "_CCM_val"],
                        columns = ["True Healthy", "False Falure", "False Healthy", "True Failure"])
    C_val['Set'] = 'Classical Validation'
    
    C_train = pd.DataFrame(data_dict[set_name + "_CCM_train"],
                        columns = ["True Healthy", "False Falure", "False Healthy", "True Failure"])
    C_train['Set'] = 'Classical Training'

    #Quantum only
    df = pd.concat([Q_val, Q_train], ignore_index=True)
    df_melted = df.melt(id_vars='Set', var_name='Metric', value_name='Value')
    
    plt.figure(figsize=(10, 6))
    sns.violinplot(x='Metric', y='Value', hue='Set', data=df_melted, split=True)
    plt.xlabel("Confusion Matrix Entries")
    plt.ylabel("Relative Frequency")
    plt.title(f'Violin Plot: Quantum Training vs Validation for {float(percent)*100}% of the data with {pca} PCA components and balanced = {balance}')
    plt.savefig(results / f"Final_Quantum_Confusion_{percent}_{pca}_bal{balance}.png")
    plt.show()
    
    #classical onlye
    df = pd.concat([C_val, C_train], ignore_index=True)
    df_melted = df.melt(id_vars='Set', var_name='Metric', value_name='Value')
    
    plt.figure(figsize=(10, 6))
    sns.violinplot(x='Metric', y='Value', hue='Set', data=df_melted, split=True)
    plt.xlabel("Confusion Matrix Entries")
    plt.ylabel("Relative Frequency")
    plt.title(f'Violin Plot: Classical Training vs Validation for {float(percent)*100}% of the data with {pca} PCA components and balanced = {balance}')
    plt.savefig(results / f"Final_Classical_Confusion_{percent}_{pca}_bal{balance}.png")
    plt.show()
    
    #all 4
    df = pd.concat([Q_val, Q_train, C_val, C_train], ignore_index=True)
    df_melted = df.melt(id_vars='Set', var_name='Metric', value_name='Value')
    
    plt.figure(figsize=(12, 6))
    sns.violinplot(x='Metric', y='Value', hue='Set', data=df_melted, split=True)
    plt.xlabel("Confusion Matrix Entries")
    plt.ylabel("Relative Frequency")
    plt.title(f'Violin Plot: Normalized results for {float(percent)*100}% of the data with {pca} PCA components and balanced = {balance}')
    plt.savefig(results / f"Final_Confusion_{percent}_{pca}_bal{balance}.png")
    plt.show()


display(widgets.interactive(
    confusion_violin,
    balance=balance_widget,
    pca=pca_widget,
    percent=percent_widget
))

interactive(children=(ToggleButtons(description='Balance:', options=('True', 'False'), value='True'), ToggleBu…

In [14]:
def final_mcc(balance, pca, percent):
    print(f"balance: {balance}, PCA Components: {pca}, Percent: {percent}")
    # Final MCC Scores
    df = pd.DataFrame(data_dict[f"{balance}_{pca}_{percent}_MCC_scores"])
    
    Q_MCC = df.iloc[:, :2].copy()
    Q_MCC.columns = ['training', 'validation']
    Q_MCC['Set'] = 'Quantum'

    C_MCC = df.iloc[:, 2:4].copy()
    C_MCC.columns = ['training', 'validation']
    C_MCC['Set'] = 'Classical'

    # Combine the data
    df = pd.concat([Q_MCC, C_MCC], ignore_index=True)
    df_melted = df.melt(id_vars='Set', var_name='Metric', value_name='Value')
    
    plt.figure(figsize=(12, 6))
    sns.violinplot(x='Metric', y='Value', hue='Set', data=df_melted, split=True)
    plt.xlabel("")
    plt.ylabel("MCC Score")
    plt.title(f'Violin Plot: MCC scores for {float(percent)*100}% of the data with {pca} PCA components and balanced = {balance}')
    #plt.savefig(results / f"FinallMCC_{percent}_{pca}_bal{balance}.png")
    plt.savefig(results / f"Final_MCCfinal_{percent}_{pca}_bal{balance}.png")
    plt.show()

display(widgets.interactive(
    final_mcc,
    balance=balance_widget,
    pca=pca_widget,
    percent=percent_widget
))

interactive(children=(ToggleButtons(description='Balance:', options=('True', 'False'), value='True'), ToggleBu…

In [18]:
balance = False
pca = 4
percent = 1

df = pd.DataFrame(data_dict[f"{balance}_{pca}_{percent}_MCC_scores"])
    
Q_MCC = df.iloc[:, :2].copy()
Q_MCC.columns = ['training', 'validation']
Q_MCC['Set'] = 'Quantum'

C_MCC = df.iloc[:, 2:4].copy()
C_MCC.columns = ['training', 'validation']
C_MCC['Set'] = 'Classical'

# Combine the data
df = pd.concat([Q_MCC, C_MCC], ignore_index=True)
df_melted = df.melt(id_vars='Set', var_name='Metric', value_name='Value')

In [26]:
df.groupby('Set').mean()

,training,validation
Set,,
Classical,0.680443,0.607848
Quantum,0.682785,0.582691
